In [159]:
import os
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_decision_forests as tfdf
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

# Set Seed cố định cho toàn bộ notebook
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

%matplotlib inline

# Check phiên bản
print("TensorFlow v" + tf.__version__)
print("TensorFlow Decision Forests v" + tfdf.__version__)

TensorFlow v2.11.0
TensorFlow Decision Forests v1.2.0


In [160]:
# 1. LOAD DỮ LIỆU
train_file_path = "../input/house-prices-advanced-regression-techniques/train.csv"
test_file_path = "../input/house-prices-advanced-regression-techniques/test.csv"

train_df = pd.read_csv(train_file_path)
test_df = pd.read_csv(test_file_path)

# Lưu lại Id của test để tạo file submission sau này
test_ids = test_df.pop('Id')
train_df = train_df.drop('Id', axis=1)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Train shape: (1460, 80), Test shape: (1459, 79)


In [161]:
# 2. ĐỊNH NGHĨA HÀM BIẾN ĐỔI FEATURE (Chỉ chứa phép toán Per-Row, Không leak)
def create_features(df):
    df = df.copy()
    
    # --- Tổng diện tích ---
    df['TotalSF']      = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath']    = df['FullBath'] + 0.5*df['HalfBath'] \
                       + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] \
                       + df['3SsnPorch'] + df['ScreenPorch']
    
    # --- Tuổi nhà ---
    df['Age']      = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['IsRemod']  = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    
    # --- Cờ nhị phân ---
    df['HasPool']      = (df['PoolArea'] > 0).astype(int)
    df['Has2ndFloor']  = (df['2ndFlrSF'] > 0).astype(int)
    df['HasGarage']    = (df['GarageArea'] > 0).astype(int)
    df['HasBsmt']      = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)

    # --- Nâng cao ---
    df['QualArea']    = df['OverallQual'] * df['GrLivArea']
    df['BathPerBed']  = df['TotalBath'] / (df['BedroomAbvGr'] + 1)
    df['SqFtPerRoom'] = df['GrLivArea'] / (df['TotRmsAbvGrd'] + 1)
    df['PorchRatio']  = df['TotalPorchSF'] / (df['TotalSF'] + 1)
    df['NewHouse']    = (df['YrSold'] == df['YearBuilt']).astype(int)
    df['TotalFin']    = df['BsmtFinSF1'] + df['BsmtFinSF2']
    df['AvgRoomSF']   = df['GrLivArea'] / df['TotRmsAbvGrd'].clip(lower=1)
    
    return df

In [162]:
# 3. TIỀN XỬ LÝ CƠ BẢN (Không phụ thuộc thống kê tập dữ liệu - An toàn với Leakage)
def basic_cleaning(df):
    df = df.copy()
    
    # Xử lý các cột Categorical NaN -> 'None'
    none_cols = [
        'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
        'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
        'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
    ]
    for col in none_cols:
        if col in df.columns:
            df[col] = df[col].fillna('None')
            
    # Điền diện tích MasVnr missing bằng 0
    if 'MasVnrArea' in df.columns:
        df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
        
    return df

def convert_types(df):
    df = df.copy()
    for col in ['MSSubClass', 'MoSold', 'YrSold']:
        if col in df.columns:
            df[col] = df[col].astype(str)
    return df

train_df = basic_cleaning(train_df)
test_df = basic_cleaning(test_df)

# Biến đổi Target Log1p
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

In [163]:
# 4. HÀM FIT & TRANSFORM IMPUTATION (Phòng chống Data Leakage triệt để)
def fit_imputer(train_data):
    """Tính toán thống kê (Median, Mode) ĐỘC LẬP trên tập train_fold."""
    electrical_mode = train_data['Electrical'].mode()[0] if not train_data['Electrical'].mode().empty else 'SBrkr'
    lot_frontage_medians = train_data.groupby('Neighborhood')['LotFrontage'].median()
    global_lot_median = train_data['LotFrontage'].median()
    
    return {
        'electrical_mode': electrical_mode,
        'lot_frontage_medians': lot_frontage_medians,
        'global_lot_median': global_lot_median
    }

def transform_imputer(df, imputer_params):
    """Điền dữ liệu thiếu dựa trên thống kê đã fit từ tập train."""
    df = df.copy()
    
    # Electrical
    if 'Electrical' in df.columns:
        df['Electrical'] = df['Electrical'].fillna(imputer_params['electrical_mode'])
        
    # LotFrontage theo Neighborhood
    if 'LotFrontage' in df.columns:
        def fill_lot(group):
            neigh_name = group.name
            median_val = imputer_params['lot_frontage_medians'].get(
                neigh_name, imputer_params['global_lot_median']
            )
            return group.fillna(median_val)
        
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(fill_lot)
        
    return df

In [164]:
# 5. ĐỊNH NGHĨA TF-DF MODELS
def make_models(seed):
    return {
        "rf": tfdf.keras.RandomForestModel(
            task=tfdf.keras.Task.REGRESSION,
            num_trees=1000,
            max_depth=20,
            min_examples=10,
            growing_strategy="BEST_FIRST_GLOBAL",
            random_seed=seed,
            verbose=0
        ),
        "gbt": tfdf.keras.GradientBoostedTreesModel(
            task=tfdf.keras.Task.REGRESSION,
            num_trees=1000,
            max_depth=6,
            shrinkage=0.01,  # Dùng shrinkage thay vì learning_rate
            random_seed=seed,
            verbose=0
        ),
        "cart": tfdf.keras.CartModel(
            task=tfdf.keras.Task.REGRESSION,
            max_depth=15,
            min_examples=5,
            random_seed=seed,
            verbose=0
        ),
    }

MODEL_NAMES = ["rf", "gbt", "cart"]

In [165]:
# 6. RUN CROSS VALIDATION & INFERENCE ON TEST
label = 'SalePrice'
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

oof = {m: np.zeros(len(train_df)) for m in MODEL_NAMES}
test_preds = {m: [] for m in MODEL_NAMES}

for fold, (tr_idx, va_idx) in enumerate(kf.split(train_df)):
    print(f"=== Fold {fold+1}/5 ===")
    
    # Tách dữ liệu thô
    tr_pd = train_df.iloc[tr_idx].reset_index(drop=True)
    va_pd = train_df.iloc[va_idx].reset_index(drop=True)
    te_pd = test_df.copy()
    
    # A. LOẠI OUTLIER CHỈ TRÊN TRAIN FOLD (Không chạm tới Valid Fold)
    tr_pd = tr_pd[~((tr_pd['GrLivArea'] > 4000) & (np.expm1(tr_pd['SalePrice']) < 300000))].reset_index(drop=True)
    
    # B. FIT IMPUTER TRÊN TRAIN FOLD & TRANSFORM CHO TRAIN, VALID, TEST
    imputer_params = fit_imputer(tr_pd)
    tr_pd = transform_imputer(tr_pd, imputer_params)
    va_pd = transform_imputer(va_pd, imputer_params)
    te_pd = transform_imputer(te_pd, imputer_params)
    
    # 2. TẠO FEATURE TRƯỚC (khi YrSold vẫn là kiểu int/float)
    tr_pd = create_features(tr_pd)
    va_pd = create_features(va_pd)
    te_pd = create_features(te_pd)
    
    # 3. CHUYỂN DẠNG STRING SAU (để TF-DF hiểu đây là Categorical feature)
    tr_pd = convert_types(tr_pd)
    va_pd = convert_types(va_pd)
    te_pd = convert_types(te_pd)
    
    # D. CHUYỂN DẠNG TF-DATASET
    tr_ds = tfdf.keras.pd_dataframe_to_tf_dataset(tr_pd, label=label, task=tfdf.keras.Task.REGRESSION)
    va_ds = tfdf.keras.pd_dataframe_to_tf_dataset(va_pd, label=label, task=tfdf.keras.Task.REGRESSION)
    te_ds = tfdf.keras.pd_dataframe_to_tf_dataset(te_pd, task=tfdf.keras.Task.REGRESSION)
    
    # E. TRAIN VÀ PREDICT CHO TỪNG MODEL
    models = make_models(seed=SEED + fold)
    for m_name, model in models.items():
        model.compile(metrics=["mse"])
        model.fit(tr_ds)
        
        # Predict Valid & Test
        val_pred = model.predict(va_ds, verbose=0).flatten()
        test_pred = model.predict(te_ds, verbose=0).flatten()
        
        oof[m_name][va_idx] = val_pred
        test_preds[m_name].append(test_pred)
        
        fold_rmsle = np.sqrt(mean_squared_error(va_pd[label], val_pred))
        print(f"  {m_name:5s} fold RMSLE = {fold_rmsle:.5f}")

# Average predictions theo Fold cho tập Test
test_preds = {m: np.mean(np.stack(v), axis=0) for m, v in test_preds.items()}

=== Fold 1/5 ===


[INFO 2026-09-17T16:02:36.857469372+00:00 kernel.cc:1214] Loading model from path /tmp/tmp3mwzzgdb/model/ with prefix 774a0013bfec4a85
[INFO 2026-09-17T16:02:37.034140839+00:00 decision_forest.cc:661] Model loaded with 1000 root(s), 61000 node(s), and 85 input feature(s).
[INFO 2026-09-17T16:02:37.034199389+00:00 abstract_model.cc:1311] Engine "RandomForestOptPred" built
[INFO 2026-09-17T16:02:37.034237539+00:00 kernel.cc:1046] Use fast generic engine


  rf    fold RMSLE = 0.14903


[INFO 2026-09-17T16:02:56.350328798+00:00 kernel.cc:1214] Loading model from path /tmp/tmp3rqnftev/model/ with prefix d48447d1c2dc44b7
[INFO 2026-09-17T16:02:56.439274437+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-09-17T16:02:56.439717507+00:00 kernel.cc:1046] Use fast generic engine


  gbt   fold RMSLE = 0.13723


...

  rf    fold RMSLE = 0.12574


[INFO 2026-09-17T16:04:51.922677072+00:00 kernel.cc:1214] Loading model from path /tmp/tmpj6mdawni/model/ with prefix 6562263bf9fc4101
[INFO 2026-09-17T16:04:52.007416301+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-09-17T16:04:52.007647231+00:00 kernel.cc:1046] Use fast generic engine


  gbt   fold RMSLE = 0.11959


[INFO 2026-09-17T16:04:54.55318556+00:00 kernel.cc:1214] Loading model from path /tmp/tmp8y1f3ng5/model/ with prefix 0de76462c8274641
[INFO 2026-09-17T16:04:54.554103209+00:00 decision_forest.cc:661] Model loaded with 1 root(s), 105 node(s), and 26 input feature(s).
[INFO 2026-09-17T16:04:54.554139889+00:00 kernel.cc:1046] Use fast generic engine


  cart  fold RMSLE = 0.17598


In [166]:
# 7. ĐÁNH GIÁ ĐỘ LỆCH OOF VÀ TÌM TRỌNG SỐ TỐI ƯU (ENSEMBLE)
y_true = train_df[label].values

print("\n=== OOF EVALUATION (RMSLE) ===")
for m in MODEL_NAMES:
    print(f"{m:5s} OOF RMSLE = {np.sqrt(mean_squared_error(y_true, oof[m])):.5f}")

def loss(w):
    w = np.abs(w)
    w = w / w.sum()
    blend = sum(w[i] * oof[m] for i, m in enumerate(MODEL_NAMES))
    return np.sqrt(mean_squared_error(y_true, blend))

res = minimize(loss, x0=np.ones(len(MODEL_NAMES))/len(MODEL_NAMES), method="Nelder-Mead")
w_opt = np.abs(res.x)
w_opt /= w_opt.sum()

print("\nOptimal weights:", dict(zip(MODEL_NAMES, np.round(w_opt, 4))))
print(f"Blend OOF RMSLE: {res.fun:.5f}")


=== OOF EVALUATION (RMSLE) ===
rf    OOF RMSLE = 0.14340
gbt   OOF RMSLE = 0.13684
cart  OOF RMSLE = 0.18370

Optimal weights: {'rf': 0.2213, 'gbt': 0.7216, 'cart': 0.0571}
Blend OOF RMSLE: 0.13563


In [167]:
# 8. SUBMISSION LOGIC
blend_log = sum(w_opt[i] * test_preds[m] for i, m in enumerate(MODEL_NAMES))
final_preds = np.expm1(blend_log) # Đảo ngược log1p

submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_preds
})

submission_df.to_csv('submission.csv', index=False)
print("\nFile 'submission.csv' đã được khởi tạo thành công!")
submission_df.head()


File 'submission.csv' đã được khởi tạo thành công!


,Id,SalePrice
0,1461,122880.648438
1,1462,157681.984375
2,1463,181947.468750
3,1464,184071.328125
4,1465,192166.093750
